# Introduction to Agent Protocols

An **agent protocol** is a standardized way for agents, tools, and services to **discover**, **communicate**, and **invoke** each other across process and organizational boundaries. Without protocols, every integration is bespoke: custom JSON shapes, ad-hoc auth, and fragile one-off clients.

Protocols answer questions like:

- How does an agent **find** what tools exist on a server?
- What **message format** carries a tool call and its result?
- How do two agents **delegate tasks** to each other?
- Where does **authentication** and **capability scoping** live?

This notebook introduces agent protocols conceptually and implements a **ShopCo Protocol Lab** — in-memory simulations of protocol patterns used by **MCP** (Model Context Protocol) and **A2A** (Agent-to-Agent). No external servers or CLI required.

In [ ]:
"""
ShopCo Protocol Lab — in-memory protocol simulation.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

PROTOCOL_LOG: list[dict[str, Any]] = []

print("ShopCo Protocol Lab ready")

---

## 1. The Interop Problem

Without a protocol, integrating agents and tools looks like this:

```
Agent A ──custom HTTP──► Service X
Agent A ──gRPC──────────► Service Y
Agent B ──Python import─► Service X  (different client!)
Agent C ──webhook───────► Service Z
```

Every pair needs a custom adapter. Protocols replace N×M adapters with **N + M** standard endpoints.

| Without protocol | With protocol |
|------------------|---------------|
| Custom message per integration | Standard envelope |
| Manual capability docs | Machine-readable discovery |
| Auth per service | Scoped connections |
| Hard to swap tools | Plug-compatible servers |

---

## 2. Protocol vs Agent Loop vs Framework

| Layer | Responsibility | Example |
|-------|----------------|--------|
| **Agent loop** | Model decides → tool executes → observe | ReAct cycle |
| **Framework** | Graph, state, checkpoints | LangGraph, CrewAI |
| **Protocol** | Wire format, discovery, transport | MCP, A2A |

Protocols do **not** replace your agent loop. They standardize how the loop **reaches** tools and other agents outside your process.

```
Agent loop (in your app)
       │
       ▼
Protocol client  ──standard messages──►  Protocol server (tools / other agent)
```

---

## 3. Protocol Stack Layers

| Layer | Purpose | MCP example | A2A example |
|-------|---------|-------------|-------------|
| **Transport** | Bytes on the wire | stdio, HTTP/SSE | HTTP + JSON-RPC |
| **Discovery** | What capabilities exist? | `tools/list` | Agent Card |
| **Invocation** | Execute capability | `tools/call` | `tasks/send` |
| **Auth / scope** | Who may call what? | Per-server connection | Agent credentials |

This notebook implements **discovery** and **invocation** in memory — transport is a Python method call instead of HTTP.

---

## 4. Standard Message Envelope

Protocols wrap payloads in a consistent envelope so clients and servers agree on structure.

In [ ]:
class MessageKind(str, Enum):
    REQUEST = "request"
    RESPONSE = "response"
    ERROR = "error"
    NOTIFICATION = "notification"


@dataclass
class ProtocolMessage:
    id: str
    method: str
    kind: MessageKind
    params: dict[str, Any] = field(default_factory=dict)
    result: Any = None
    error: str | None = None
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    )

    def to_dict(self) -> dict[str, Any]:
        d: dict[str, Any] = {
            "jsonrpc": "2.0",
            "id": self.id,
            "method": self.method,
            "kind": self.kind.value,
            "timestamp": self.timestamp,
        }
        if self.kind == MessageKind.REQUEST:
            d["params"] = self.params
        elif self.kind == MessageKind.RESPONSE:
            d["result"] = self.result
        elif self.kind == MessageKind.ERROR:
            d["error"] = {"message": self.error}
        return d


req = ProtocolMessage(
    id="msg-001",
    method="tools/call",
    kind=MessageKind.REQUEST,
    params={"name": "lookup_order", "arguments": {"order_id": "ORD-1001"}},
)
print(pretty(req.to_dict()))

---

## 5. Capability Discovery

Before calling a tool, the client asks: **what can this server do?** Discovery returns machine-readable schemas.

In [ ]:
@dataclass
class ToolCapability:
    name: str
    description: str
    input_schema: dict[str, Any]
    risk_tier: str = "read"


@dataclass
class ResourceCapability:
    uri: str
    name: str
    description: str
    mime_type: str = "application/json"


SHOPCO_TOOL_CAPABILITIES = [
    ToolCapability(
        name="lookup_order",
        description="Look up order status by ORD-#### id",
        input_schema={
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    ),
    ToolCapability(
        name="search_policy",
        description="Search return and cancellation policies",
        input_schema={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    ),
]

SHOPCO_RESOURCES = [
    ResourceCapability(
        uri="shopco://policies/returns",
        name="returns_policy",
        description="Current returns policy document",
    ),
]

print("Discovered tools:")
for cap in SHOPCO_TOOL_CAPABILITIES:
    print(f"  • {cap.name}: {cap.description}")

---

## 6. Protocol Server — MCP-Style Pattern

An MCP **server** exposes tools and resources. It handles `tools/list`, `tools/call`, and `resources/read` requests.

In [ ]:
def _lookup_order_impl(order_id: str) -> dict[str, Any]:
    oid = order_id.upper()
    order = ORDERS.get(oid)
    if not order:
        return {"found": False, "order_id": oid}
    return {"found": True, "order_id": oid, **order}


def _search_policy_impl(query: str) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    return [p for p in POLICIES if any(t in p["text"].lower() for t in terms)] or POLICIES[:1]


TOOL_HANDLERS: dict[str, Callable[..., Any]] = {
    "lookup_order": _lookup_order_impl,
    "search_policy": _search_policy_impl,
}


class ShopCoProtocolServer:
    """In-memory MCP-style server."""

    def __init__(self, name: str = "shopco-support"):
        self.name = name
        self.tools = SHOPCO_TOOL_CAPABILITIES
        self.resources = SHOPCO_RESOURCES

    def handle(self, msg: ProtocolMessage) -> ProtocolMessage:
        PROTOCOL_LOG.append({"direction": "in", "method": msg.method, "id": msg.id})

        if msg.method == "tools/list":
            result = [
                {"name": t.name, "description": t.description, "inputSchema": t.input_schema}
                for t in self.tools
            ]
            return self._respond(msg.id, msg.method, result)

        if msg.method == "tools/call":
            name = msg.params.get("name", "")
            args = msg.params.get("arguments", {})
            handler = TOOL_HANDLERS.get(name)
            if handler is None:
                return self._error(msg.id, msg.method, f"Unknown tool: {name}")
            try:
                result = handler(**args)
                return self._respond(msg.id, msg.method, {"content": [{"type": "text", "text": json.dumps(result)}]})
            except Exception as exc:
                return self._error(msg.id, msg.method, str(exc))

        if msg.method == "resources/list":
            result = [{"uri": r.uri, "name": r.name, "description": r.description} for r in self.resources]
            return self._respond(msg.id, msg.method, result)

        if msg.method == "resources/read":
            uri = msg.params.get("uri", "")
            if uri == "shopco://policies/returns":
                return self._respond(msg.id, msg.method, {"contents": [{"uri": uri, "text": pretty(POLICIES)}]})
            return self._error(msg.id, msg.method, f"Unknown resource: {uri}")

        return self._error(msg.id, msg.method, f"Unknown method: {msg.method}")

    def _respond(self, req_id: str, method: str, result: Any) -> ProtocolMessage:
        resp = ProtocolMessage(id=req_id, method=method, kind=MessageKind.RESPONSE, result=result)
        PROTOCOL_LOG.append({"direction": "out", "method": method, "id": req_id, "ok": True})
        return resp

    def _error(self, req_id: str, method: str, message: str) -> ProtocolMessage:
        resp = ProtocolMessage(id=req_id, method=method, kind=MessageKind.ERROR, error=message)
        PROTOCOL_LOG.append({"direction": "out", "method": method, "id": req_id, "ok": False})
        return resp


server = ShopCoProtocolServer()
print(f"Server '{server.name}' ready with {len(server.tools)} tools")

---

## 7. Protocol Client — Discovery and Invocation

An MCP **host** embeds a **client** that connects to servers, discovers tools, and forwards agent tool calls.

In [ ]:
class ShopCoProtocolClient:
    """In-memory MCP-style client."""

    def __init__(self, server: ShopCoProtocolServer):
        self._server = server
        self._tool_cache: list[dict[str, Any]] | None = None

    def _request(self, method: str, params: dict | None = None) -> ProtocolMessage:
        msg = ProtocolMessage(
            id=str(uuid.uuid4())[:8],
            method=method,
            kind=MessageKind.REQUEST,
            params=params or {},
        )
        return self._server.handle(msg)

    def list_tools(self) -> list[dict[str, Any]]:
        if self._tool_cache is None:
            resp = self._request("tools/list")
            self._tool_cache = resp.result if resp.kind == MessageKind.RESPONSE else []
        return self._tool_cache

    def call_tool(self, name: str, arguments: dict[str, Any]) -> Any:
        resp = self._request("tools/call", {"name": name, "arguments": arguments})
        if resp.kind == MessageKind.ERROR:
            raise RuntimeError(resp.error)
        content = resp.result.get("content", [{}])[0].get("text", "{}")
        return json.loads(content)

    def read_resource(self, uri: str) -> str:
        resp = self._request("resources/read", {"uri": uri})
        if resp.kind == MessageKind.ERROR:
            raise RuntimeError(resp.error)
        return resp.result["contents"][0]["text"]

    def tools_for_llm(self) -> list[dict[str, Any]]:
        """Convert discovered tools to OpenAI-style function schemas."""
        return [
            {
                "type": "function",
                "function": {
                    "name": t["name"],
                    "description": t["description"],
                    "parameters": t["inputSchema"],
                },
            }
            for t in self.list_tools()
        ]


PROTOCOL_LOG.clear()
client = ShopCoProtocolClient(server)

print("Discovered tools:", [t["name"] for t in client.list_tools()])
order = client.call_tool("lookup_order", {"order_id": "ORD-1001"})
print("lookup_order result:", order)

---

## 8. Agent Loop Through a Protocol Client

The agent loop stays the same — only the **tool execution layer** goes through the protocol.

In [ ]:
class ProtocolBackedAgent:
    """Simple agent that routes tool calls through ShopCoProtocolClient."""

    def __init__(self, client: ShopCoProtocolClient):
        self._client = client

    def run(self, user_query: str) -> dict[str, Any]:
        trace: list[str] = []
        observations: list[Any] = []

        if "ord-" in user_query.lower():
            match = re.search(r"ORD-\d{4}", user_query.upper())
            oid = match.group(0) if match else "ORD-1001"
            obs = self._client.call_tool("lookup_order", {"order_id": oid})
            observations.append(obs)
            trace.append(f"tools/call lookup_order({oid})")

        if any(w in user_query.lower() for w in ("return", "policy", "cancel")):
            obs = self._client.call_tool("search_policy", {"query": user_query[:40]})
            observations.append(obs)
            trace.append("tools/call search_policy")

        if not observations:
            policy_text = self._client.read_resource("shopco://policies/returns")
            observations.append(json.loads(policy_text))
            trace.append("resources/read shopco://policies/returns")

        answer = f"ShopCo support: {json.dumps(observations, default=str)[:200]}..."
        return {"answer": answer, "trace": trace, "tools_available": len(self._client.list_tools())}


agent = ProtocolBackedAgent(client)
result = agent.run("Where is order ORD-1001 and what is the return policy?")
print(result["answer"][:120], "...")
print("Trace:", result["trace"])

---

## 9. Agent-to-Agent (A2A) — Protocol for Delegation

**MCP** connects agents to **tools**. **A2A** connects agents to **other agents** — task delegation across services or organizations.

| MCP | A2A |
|-----|-----|
| Agent → Tool | Agent → Agent |
| `tools/list`, `tools/call` | Agent Card, `tasks/send` |
| Capability = function | Capability = another agent |
| ShopCo order API | External billing specialist agent |

In [ ]:
@dataclass
class AgentCard:
    """A2A-style agent metadata for discovery."""
    agent_id: str
    name: str
    description: str
    skills: list[str]
    input_modes: list[str] = field(default_factory=lambda: ["text"])
    output_modes: list[str] = field(default_factory=lambda: ["text"])
    endpoint: str = ""


@dataclass
class A2ATask:
    task_id: str
    from_agent: str
    to_agent: str
    message: str
    context: dict[str, Any] = field(default_factory=dict)
    status: str = "pending"
    result: str = ""


SUPPORT_AGENT_CARD = AgentCard(
    agent_id="shopco-support-v1",
    name="ShopCo Support Agent",
    description="Handles order lookups and policy questions",
    skills=["order_lookup", "policy_search"],
    endpoint="shopco://agents/support",
)

BILLING_AGENT_CARD = AgentCard(
    agent_id="shopco-billing-v1",
    name="ShopCo Billing Agent",
    description="Handles invoice disputes and refunds",
    skills=["billing_lookup", "refund_policy"],
    endpoint="shopco://agents/billing",
)

AGENT_REGISTRY: dict[str, AgentCard] = {
    SUPPORT_AGENT_CARD.agent_id: SUPPORT_AGENT_CARD,
    BILLING_AGENT_CARD.agent_id: BILLING_AGENT_CARD,
}

print("Registered agents:")
for card in AGENT_REGISTRY.values():
    print(f"  • {card.agent_id}: {card.skills}")

---

## 10. A2A Task Broker — Simulated Delegation

A coordinator agent discovers remote agents via **Agent Cards** and sends **tasks** instead of calling tools directly.

In [ ]:
class A2ABroker:
    """Simulates agent-to-agent task routing."""

    def __init__(self):
        self.tasks: list[A2ATask] = []
        self._handlers: dict[str, Callable[[A2ATask], str]] = {
            "shopco-support-v1": self._handle_support,
            "shopco-billing-v1": self._handle_billing,
        }

    def discover(self, skill: str) -> list[AgentCard]:
        return [c for c in AGENT_REGISTRY.values() if skill in c.skills]

    def send_task(self, from_agent: str, to_agent_id: str, message: str, **ctx: Any) -> A2ATask:
        task = A2ATask(
            task_id=str(uuid.uuid4())[:8],
            from_agent=from_agent,
            to_agent=to_agent_id,
            message=message,
            context=dict(ctx),
        )
        handler = self._handlers.get(to_agent_id)
        if handler is None:
            task.status = "failed"
            task.result = f"Unknown agent: {to_agent_id}"
        else:
            task.result = handler(task)
            task.status = "completed"
        self.tasks.append(task)
        return task

    def _handle_support(self, task: A2ATask) -> str:
        c = ShopCoProtocolClient(server)
        if "ord-" in task.message.lower():
            match = re.search(r"ORD-\d{4}", task.message.upper())
            oid = match.group(0) if match else "ORD-1001"
            data = c.call_tool("lookup_order", {"order_id": oid})
            return f"Support: order {data.get('order_id')} is {data.get('status', 'unknown')}"
        policies = c.call_tool("search_policy", {"query": task.message[:40]})
        return f"Support: {policies[0]['text']} [{policies[0]['id']}]"

    def _handle_billing(self, task: A2ATask) -> str:
        email = task.context.get("email", "customer@shop.com")
        return f"Billing: dispute opened for {email} — resolved within 5 business days [bill-04]"


broker = A2ABroker()

# Coordinator routes by skill
user_query = "I was charged twice on my invoice"
billing_agents = broker.discover("billing_lookup")
target = billing_agents[0].agent_id if billing_agents else "shopco-support-v1"

task = broker.send_task(
    from_agent="coordinator",
    to_agent_id=target,
    message=user_query,
    email="alice@shop.com",
)
print(f"Task {task.task_id}: {task.status}")
print(f"Result: {task.result}")

---

## 11. MCP + A2A Together

Real systems combine both: an agent uses **MCP** for tools and **A2A** to delegate to specialist agents.

```
                    Coordinator Agent
                           │
            ┌──────────────┼──────────────┐
            ▼              ▼              ▼
      MCP: orders     A2A: billing    MCP: policies
      (tool server)   (remote agent)  (resource server)
```

In [ ]:
def hybrid_coordinator(query: str) -> dict[str, Any]:
    """Uses MCP for tools and A2A for billing delegation."""
    trace: list[str] = []
    parts: list[str] = []
    mcp = ShopCoProtocolClient(server)
    a2a = A2ABroker()

    if "ord-" in query.lower():
        match = re.search(r"ORD-\d{4}", query.upper())
        if match:
            data = mcp.call_tool("lookup_order", {"order_id": match.group(0)})
            parts.append(f"Order: {data.get('status')}")
            trace.append("MCP tools/call lookup_order")

    if any(w in query.lower() for w in ("charge", "invoice", "bill", "refund")):
        agents = a2a.discover("billing_lookup")
        if agents:
            task = a2a.send_task("coordinator", agents[0].agent_id, query)
            parts.append(task.result)
            trace.append(f"A2A tasks/send → {agents[0].agent_id}")

    if any(w in query.lower() for w in ("return", "policy")):
        hits = mcp.call_tool("search_policy", {"query": query[:40]})
        parts.append(f"Policy [{hits[0]['id']}]: {hits[0]['text']}")
        trace.append("MCP tools/call search_policy")

    return {
        "answer": " | ".join(parts) if parts else "How can I help?",
        "trace": trace,
    }


for q in [
    "Where is ORD-1001?",
    "I was charged twice on my invoice",
    "ORD-1002 cancel policy?",
]:
    out = hybrid_coordinator(q)
    print(f"\nQ: {q}")
    print(f"A: {out['answer']}")
    print(f"   {out['trace']}")

---

## 12. Ad-Hoc vs Protocol Integration

| | Ad-hoc | Protocol |
|--|--------|----------|
| Discovery | Read README | `tools/list` / Agent Card |
| Invocation | Custom HTTP POST | `tools/call` / `tasks/send` |
| Schema | Hope it matches | `inputSchema` in manifest |
| Swap backend | Rewrite client | Connect different server |
| Cross-org | Painful | Designed for it (A2A) |

In [ ]:
def ad_hoc_integration(order_id: str) -> dict:
    """Brittle: hardcoded URL shape (anti-pattern demo)."""
    return {"url": f"/api/v2/orders/{order_id}", "headers": {"X-Custom": "yes"}}


def protocol_integration(order_id: str) -> ProtocolMessage:
    """Standard: discover once, call by name."""
    return ProtocolMessage(
        id="1", method="tools/call", kind=MessageKind.REQUEST,
        params={"name": "lookup_order", "arguments": {"order_id": order_id}},
    )


print("Ad-hoc:", ad_hoc_integration("ORD-1001"))
print("Protocol:", pretty(protocol_integration("ORD-1001").to_dict()))

---

## 13. When You Need a Protocol

| Use a protocol when… | Skip protocols when… |
|----------------------|---------------------|
| Tools live in another process/repo | All tools are in-process Python |
| Multiple agents share tool servers | Single app, single agent |
| Cross-team or cross-org integration | Prototype with 2 tools |
| You need discoverable capability manifests | Tools are fixed at build time |
| Compliance requires audited boundaries | Internal script, no production |

---

## 14. Security at the Protocol Boundary

| Risk | Mitigation |
|------|------------|
| Over-privileged server connection | Per-server scopes, least privilege |
| Untrusted tool observations | Treat as untrusted input to model |
| Agent impersonation (A2A) | Verify Agent Card credentials |
| SSRF via resource URIs | Allowlist URI schemes |
| Secret leakage in logs | Redact protocol message payloads |

In [ ]:
@dataclass
class ConnectionScope:
    server_name: str
    allowed_tools: list[str]
    allowed_resources: list[str]


SUPPORT_SCOPE = ConnectionScope(
    server_name="shopco-support",
    allowed_tools=["lookup_order", "search_policy"],
    allowed_resources=["shopco://policies/returns"],
)


def scoped_call(client: ShopCoProtocolClient, scope: ConnectionScope, name: str, args: dict) -> Any:
    if name not in scope.allowed_tools:
        raise PermissionError(f"Tool '{name}' not in scope for {scope.server_name}")
    return client.call_tool(name, args)


print(scoped_call(client, SUPPORT_SCOPE, "lookup_order", {"order_id": "ORD-1001"}))
try:
    scoped_call(client, SUPPORT_SCOPE, "delete_order", {"order_id": "ORD-1001"})
except PermissionError as e:
    print(f"Blocked: {e}")

---

## 15. Protocol Message Log

Inspect the wire-level conversation from our in-memory simulation.

In [ ]:
print(f"Protocol log ({len(PROTOCOL_LOG)} entries):")
for entry in PROTOCOL_LOG[-10:]:
    print(f"  {entry}")

---

## 16. Major Protocols at a Glance

| Protocol | Focus | Key primitives |
|----------|-------|----------------|
| **MCP** | Agent ↔ Tools/Resources | Host, Client, Server; tools/list, tools/call |
| **A2A** | Agent ↔ Agent | Agent Card; tasks/send, message exchange |
| **OpenAPI** | HTTP API contracts | REST paths (often wrapped by MCP) |
| **JSON-RPC 2.0** | Message envelope | Often the payload shape under MCP/A2A |

---

## 17. Check Your Understanding

1. What problem do agent protocols solve compared to ad-hoc integrations?
2. How does a **protocol** differ from an **agent loop**?
3. What are the four layers of a protocol stack?
4. What does MCP standardize vs what does A2A standardize?
5. What happens during **capability discovery**?
6. Why should tool observations from a protocol still be treated as untrusted?
7. When would you skip protocols and use in-process tools?

---

## 18. Summary

Agent protocols standardize how agents reach **tools** and **other agents** outside their process:

- **Discovery** — machine-readable capability manifests (`tools/list`, Agent Cards).
- **Invocation** — standard messages (`tools/call`, `tasks/send`).
- **Envelope** — JSON-RPC-style requests, responses, and errors.
- **MCP** — agent-to-tool; **A2A** — agent-to-agent; often used together.
- **Security** — connection scopes, least privilege, untrusted observations.

The ShopCo Protocol Lab simulated an MCP server/client and an A2A broker in plain Python. Your agent loop stays the same — protocols replace bespoke adapters at the boundary.